# 🏷️ Notebook 03 — Medical Category Labelling
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the cleaned dataset (`pubmedqa_cleaned.csv`)
- Apply keyword-regex labelling to assign one of 6 medical categories
- Validate all 6 categories are present with ≥ 1% representation each
- Save the labelled dataset as `pubmedqa_labelled.csv`

### 📋 Output
- `data/processed/pubmedqa_labelled.csv` — dataset with `category` column

### 📌 Category Definitions
| Category | Description |
|---|---|
| Symptoms | Questions about signs, pain, manifestations |
| Diagnosis | Questions about detection, screening, assessment |
| Treatment | Questions about therapies, surgeries, procedures |
| Medication | Questions about drugs, dosages, vaccines |
| Prevention | Questions about risk reduction, lifestyle, avoidance |
| General | Everything else |

---

## 1. Imports & Configuration

In [1]:
import pandas as pd
import os
import sys

# Add project root to path so we can import src modules
sys.path.append(os.path.abspath('..'))

from src.data.labeller import label_dataframe, print_category_distribution

print('✅ Imports successful')

✅ Imports successful


## 2. Load Cleaned Dataset

In [2]:
input_path = '../data/processed/pubmedqa_cleaned.csv'

df = pd.read_csv(input_path)

print(f'✅ Loaded: {input_path}')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns: {df.columns.tolist()}')

# Verify expected columns exist
assert 'question' in df.columns, "Missing 'question' column"
assert 'context' in df.columns, "Missing 'context' column"
assert 'answer' in df.columns, "Missing 'answer' column"

df.head(3)

✅ Loaded: ../data/processed/pubmedqa_cleaned.csv
   Shape: 10,000 rows × 3 columns
   Columns: ['question', 'context', 'answer']


,question,context,answer
0,Is naturopathy as effective as conventional th...,Although the use of alternative medicine in th...,Naturopathy appears to be an effective alterna...
1,Can randomised trials rely on existing electro...,"To estimate the feasibility, utility and resou...",Routine data have the potential to support hea...
2,Is laparoscopic radical prostatectomy better t...,To compare morbidity in two groups of patients...,The results of our non-randomized study show t...


## 3. Apply Category Labelling

We use `src/data/labeller.py` which applies keyword-regex matching
to the combined `question + context` text for each row.

In [3]:
print('⏳ Applying category labels...')

df = label_dataframe(
    df,
    question_col='question',
    context_col='context',
    output_col='category'
)

print('✅ Labelling complete!')
print(f'   New columns: {df.columns.tolist()}')

⏳ Applying category labels...
✅ Labelling complete!
   New columns: ['question', 'context', 'answer', 'category']


## 4. Validate Category Distribution

In [4]:
counts = print_category_distribution(df, col='category')


Category          Count Percentage
───────────────────────────────────
Symptoms          5,893      58.9%
Diagnosis         3,043      30.4%
Treatment           330       3.3%
Medication          295       2.9%
General             290       2.9%
Prevention          149       1.5%

Total rows: 10,000
Categories: 6

✅ All categories have ≥ 1% representation


## 5. KPI Check — All 6 Categories ≥ 1%

In [5]:
total = len(df)
n_categories = df['category'].nunique()
min_pct = (df['category'].value_counts() / total * 100).min()

print(f'\n📊 KPI Validation:')
print(f'   Categories found: {n_categories} / 6 expected')
print(f'   Minimum category %: {min_pct:.2f}%')

assert n_categories == 6, f"Expected 6 categories, found {n_categories}"
assert min_pct >= 1.0, f"Category below 1%: {min_pct:.2f}%"

print('   ✅ All 6 categories present with ≥ 1% representation')


📊 KPI Validation:
   Categories found: 6 / 6 expected
   Minimum category %: 1.49%
   ✅ All 6 categories present with ≥ 1% representation


In [6]:
# Add this cell BEFORE the labelling step
import importlib
import src.data.labeller as labeller_module

# Force Python to reload the module (in case it cached the old version)
importlib.reload(labeller_module)

from src.data.labeller import CATEGORY_KEYWORDS

print("Prevention keywords:")
print(CATEGORY_KEYWORDS['Prevention'])
print(f"\nCount: {len(CATEGORY_KEYWORDS['Prevention'])}")

total = len(df)
for cat, count in df['category'].value_counts().items():
    print(f"{cat:<15} {count:>6} → {count/total*100:.2f}%")

Prevention keywords:
['prevent', 'protect', 'avoid', 'lifestyle', 'diet', 'exercise', 'smoking', 'obesity', 'vaccination', 'screening program', 'risk reduction', 'prophyla', 'immuniz', 'hygiene', 'safe sex', 'breastfeed', 'nutriti', 'physical activity', 'weight loss', 'risk factor', 'early detection', 'health promotion', 'public health', 'epidemiolog']

Count: 24
Symptoms          5893 → 58.93%
Diagnosis         3043 → 30.43%
Treatment          330 → 3.30%
Medication         295 → 2.95%
General            290 → 2.90%
Prevention         149 → 1.49%


## 6. Sample Rows per Category

In [7]:
pd.set_option('display.max_colwidth', 80)

for cat in sorted(df['category'].unique()):
    print(f'\n{"=" * 80}')
    print(f'Category: {cat}')
    print(f'{"=" * 80}')
    sample = df[df['category'] == cat].head(2)
    for _, row in sample.iterrows():
        print(f'  Q: {row["question"][:100]}')
        print(f'  A: {row["answer"][:100]}')
        print()


Category: Diagnosis
  Q: Management of thoracic empyema in childhood: does the pleural thickening matter?
  A: Results suggest that decortication is not necessary in children to prevent long term problems with p

  Q: Lower urinary tract reconstruction for duplicated renal units with ureterocele. Is excision of the u
  A: Lower urinary tract reconstruction for duplicated renal systems with obstruction of the upper pole c


Category: General
  Q: Do people recognise mental illness?
  A: The low knowledge about mental disorders, particularly depression, confirms the importance and the n

  Q: Can keratinocytes cause failure of osseointegration?
  A: These findings suggest that, in the cases reported, keratinocytes implanted between the titanium and


Category: Medication
  Q: Efficacy of secondary isoniazid preventive therapy among HIV-infected Southern Africans: time to cha
  A: Secondary preventive therapy reduces TB recurrence: the absolute impact appears to be greatest among

  Q: I

## 7. Save Labelled Dataset

In [8]:
output_path = '../data/processed/pubmedqa_labelled.csv'
os.makedirs('../data/processed', exist_ok=True)

# Verify final column set
expected_cols = ['question', 'context', 'answer', 'category']
assert list(df.columns) == expected_cols, \
    f"Expected {expected_cols}, got {list(df.columns)}"

df.to_csv(output_path, index=False)

print(f'✅ Labelled dataset saved to: {output_path}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.columns.tolist()}')
print(f'   File size: {os.path.getsize(output_path) / 1024**2:.1f} MB')

✅ Labelled dataset saved to: ../data/processed/pubmedqa_labelled.csv
   Rows: 10,000
   Columns: ['question', 'context', 'answer', 'category']
   File size: 16.5 MB


## ✅ Summary

| Item | Result |
|---|---|
| Input | `data/processed/pubmedqa_cleaned.csv` |
| Output | `data/processed/pubmedqa_labelled.csv` |
| New column | `category` |
| Categories | Symptoms, Diagnosis, Treatment, Medication, Prevention, General |
| All ≥ 1% | ✅ |

---

### ➡️ Next Step
Open **`03_eda.ipynb`** to explore and visualise the labelled dataset.